In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "nyc_taxi.silver.yellow_trips"
KEY = ["vendor_id", "pickup_datetime", "dropoff_datetime",
       "pickup_location_id", "dropoff_location_id"]

silver = spark.table(SILVER_TABLE)

total_rows    = silver.count()
distinct_keys = silver.select(KEY).distinct().count()

print(f"total rows    : {total_rows:,}")
print(f"distinct keys : {distinct_keys:,}")
print(f"extra rows    : {total_rows - distinct_keys:,}   (rows that share a key with another row)")

In [0]:
dup_keys = silver.groupBy(KEY).count().filter("count > 1")

display(dup_keys.groupBy("count").agg(F.count("*").alias("num_keys")).orderBy("count"))
display(dup_keys.orderBy(F.desc("count")).limit(10))

In [0]:
business_cols = [c for c in silver.columns if not c.startswith("_")]

exact_distinct = silver.select(business_cols).distinct().count()
print(f"rows             : {total_rows:,}")
print(f"distinct (all business columns): {exact_distinct:,}")
print(f"exact duplicates : {total_rows - exact_distinct:,}")

In [0]:
sample_key = dup_keys.orderBy(F.desc("count")).first()
cond = [F.col(k) == sample_key[k] for k in KEY]
display(silver.filter(cond[0] & cond[1] & cond[2] & cond[3] & cond[4]))

In [0]:
# 1) 取出所有重复键对应的行
dup_rows = silver.join(dup_keys.select(KEY), on=KEY, how="inner")

# 2) 要比较的列：业务列中除了键本身，再加上 _source_file 看两行是否来自不同文件
compare_cols = [c for c in business_cols if c not in KEY] + ["_source_file"]

def nd(c):
    # NULL 安全的"不同值个数"
    return F.count_distinct(F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))).alias(c)

per_group = dup_rows.groupBy(KEY).agg(*[nd(c) for c in compare_cols])

# 3) 每列在多少组里出现差异（>1 个不同值）
summary = per_group.select([F.sum((F.col(c) > 1).cast("int")).alias(c) for c in compare_cols])
display(summary)


In [0]:
show_cols = KEY + ["passenger_count", "payment_type", "fare_amount", "extra", "mta_tax",
                   "tip_amount", "tolls_amount", "total_amount", "trip_distance", "_source_file"]
display(dup_rows.select(show_cols).orderBy(KEY).limit(20))
